# VisionGuard: Experiment 4, YOLO11n trained with extra SH17 photos (`n_core_sh17_imgsz960`)
One change from `n_core_imgsz960`: the training data. Model, image size, epochs, batch and seed are identical.
Validation and test images are exactly the same as before, so the scores are comparable.

**Before you run:** GPU on (T4), Internet On, and add TWO inputs: `construction-ppe` (yours) and `sh17-dataset-for-ppe-detection` (by mugheesahmad).
SH17 is CC BY-NC-SA 4.0 (non-commercial). Do not publish the dataset or the notebook output folder `datasets/`. Takes about 40 minutes.

In [ ]:
!git clone https://github.com/SUBASHRI-S/visionguard.git
%cd visionguard
!pip install -q ultralytics==8.4.216 mlflow==3.18.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

Rebuild the exact base dataset. Must print `rebuilt d1dc2ab4d1a9 (core)`.

In [ ]:
!mkdir -p datasets/raw
!cp -r /kaggle/input/datasets/subaaaa/construction-ppe/construction-ppe datasets/raw/
!python -m scripts.rebuild_dataset --version d1dc2ab4d1a9

Add the selected SH17 photos to the training split only. Prints how many photos were found, dropped as near-duplicates, and added.

In [ ]:
!python -m scripts.extend_with_sh17 --sh17 /kaggle/input/datasets/mugheesahmad/sh17-dataset-for-ppe-detection

Write the training config. It is a copy of `n_core_imgsz960.yaml` with only the name, profile and dataset version changed.

In [ ]:
import json, pathlib, re
versions = [json.loads(p.read_text()) for p in pathlib.Path("datasets/versions").glob("*.json")]
version = next(m["dataset_version"] for m in versions if m["profile"] == "core_sh17")
text = pathlib.Path("experiments/configs/n_core_imgsz960.yaml").read_text()
text = re.sub(r"^name: .*$", "name: n_core_sh17_imgsz960", text, flags=re.M)
text = re.sub(r"profile: core\b", "profile: core_sh17", text)
text = re.sub(r"version: [0-9a-f]{12}", f"version: {version}", text)
text = "# Experiment 4. One change from n_core_imgsz960: SH17 photos with a helmet or vest added to TRAIN.\n" + text
pathlib.Path("experiments/configs/n_core_sh17_imgsz960.yaml").write_text(text)
print(version); print(text)

In [ ]:
!python -m scripts.train --config experiments/configs/n_core_sh17_imgsz960.yaml --no-mlflow

Score on the unchanged validation and test splits (same images as every earlier run).

In [ ]:
!python -m scripts.evaluate --run runs/n_core_sh17_imgsz960 --split val
!python -m scripts.evaluate --run runs/n_core_sh17_imgsz960 --split test

Package the run, the config and the small dataset manifest. Download `sh17_run.zip` from the Output tab. It does NOT contain any images.

In [ ]:
!zip -qr /kaggle/working/sh17_run.zip runs/n_core_sh17_imgsz960 evaluation/results datasets/versions experiments/configs/n_core_sh17_imgsz960.yaml
!ls -lh /kaggle/working